<a href="https://colab.research.google.com/github/frankettheofranckettheo/Advanced-ML-Project/blob/tp4/U_NetArchitechture_tp4.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 1. Semantic Segmentation on MedicalData

 ## Exercise1: Implementing the U-Net Architecture

In [11]:
import tensorflow as tf
from tensorflow import keras

def conv_block(input_tensor, num_filters):
    # Core convolutional block
    x = keras.layers.Conv2D(num_filters, (3, 3), padding='same')(input_tensor)
    x = keras.layers.BatchNormalization()(x)
    x = keras.layers.Activation('relu')(x)

    x = keras.layers.Conv2D(num_filters, (3, 3), padding='same')(x)
    x = keras.layers.BatchNormalization()(x)
    x = keras.layers.Activation('relu')(x)

    return x

def build_unet(input_shape=(128, 128, 1)):
    inputs = keras.Input(input_shape)

    # ENCODER PATH (Contracting)
    c1 = conv_block(inputs, 32)
    p1 = keras.layers.MaxPooling2D((2, 2))(c1)

    c2 = conv_block(p1, 64)
    p2 = keras.layers.MaxPooling2D((2, 2))(c2)

    c3 = conv_block(p2, 128)
    p3 = keras.layers.MaxPooling2D((2, 2))(c3)

    # BRIDGE / BOTTLENECK
    b = conv_block(p3, 256)
    # DECODER PATH (Expansive)

    # Step 1
    u1 = keras.layers.Conv2DTranspose(
        128, (2, 2), strides=(2, 2), padding='same'
    )(b)
    u1 = keras.layers.Concatenate()([u1, c3])
    d1 = conv_block(u1, 128)

    # Step 2
    u2 = keras.layers.Conv2DTranspose(
        64, (2, 2), strides=(2, 2), padding='same'
    )(d1)
    u2 = keras.layers.Concatenate()([u2, c2])
    d2 = conv_block(u2, 64)

    # Step 3
    u3 = keras.layers.Conv2DTranspose(
        32, (2, 2), strides=(2, 2), padding='same'
    )(d2)
    u3 = keras.layers.Concatenate()([u3, c1])
    d3 = conv_block(u3, 32)

    # Output layer
    outputs = keras.layers.Conv2D(
        1, (1, 1), activation='sigmoid'
    )(d3)

    return keras.Model(inputs=[inputs], outputs=[outputs])


## Exercice 2: Segmentation-Specific Metrics

In [14]:
import tensorflow as tf

def dice_coeff(y_true, y_pred, smooth=1.):
    # Flatten the tensors
    y_true_f = tf.reshape(y_true, [-1])
    y_pred_f = tf.reshape(y_pred, [-1])

    intersection = tf.reduce_sum(y_true_f * y_pred_f)
    return (2. * intersection + smooth) / (tf.reduce_sum(y_true_f) + tf.reduce_sum(y_pred_f) + smooth)

def iou_metric(y_true, y_pred, smooth=1.):
    # On aplatit les tenseurs
    y_true_f = tf.reshape(y_true, [-1])
    y_pred_f = tf.reshape(y_pred, [-1])

    # Intersection et union
    intersection = tf.reduce_sum(tf.abs(y_true_f * y_pred_f))
    union = tf.reduce_sum(y_true_f) + tf.reduce_sum(y_pred_f) - intersection

    return (intersection + smooth) / (union + smooth)


In [15]:
import numpy as np

# Compile and train the model using a suitable loss and custom metric (IoU/Dice)
def dice_loss(y_true, y_pred):
    return 1 - dice_coeff(y_true, y_pred)



X = np.random.rand(10, 128, 128, 1).astype("float32")
Y = np.random.randint(0, 2, (10, 128, 128, 1)).astype("float32")


model = build_unet(input_shape=(128, 128, 1))

model.compile(
    optimizer=keras.optimizers.Adam(),
    loss=dice_loss,
    metrics=[dice_coeff, iou_metric]
)

history = model.fit(
    X,
    Y,
    batch_size=16,
    epochs=20,
    validation_split=0.2,
    shuffle=True
)


Epoch 1/20
1/1 ━━━━━━━━━━━━━━━━━━━━ 19s 19s/step - dice_coeff: 0.4870 - iou_metric: 0.3219 - loss: 0.5130 - val_dice_coeff: 0.5025 - val_iou_metric: 0.3356 - val_loss: 0.4975
Epoch 2/20
1/1 ━━━━━━━━━━━━━━━━━━━━ 4s 4s/step - dice_coeff: 0.5112 - iou_metric: 0.3434 - loss: 0.4888 - val_dice_coeff: 0.5024 - val_iou_metric: 0.3355 - val_loss: 0.4976
Epoch 3/20
1/1 ━━━━━━━━━━━━━━━━━━━━ 4s 4s/step - dice_coeff: 0.5315 - iou_metric: 0.3619 - loss: 0.4685 - val_dice_coeff: 0.5025 - val_iou_metric: 0.3356 - val_loss: 0.4975
Epoch 4/20
1/1 ━━━━━━━━━━━━━━━━━━━━ 5s 5s/step - dice_coeff: 0.5504 - iou_metric: 0.3796 - loss: 0.4496 - val_dice_coeff: 0.5028 - val_iou_metric: 0.3358 - val_loss: 0.4972
Epoch 5/20
1/1 ━━━━━━━━━━━━━━━━━━━━ 4s 4s/step - dice_coeff: 0.5685 - iou_metric: 0.3972 - loss: 0.4315 - val_dice_coeff: 0.5035 - val_iou_metric: 0.3364 - val_loss: 0.4965
Epoch 6/20
1/1 ━━━━━━━━━━━━━━━━━━━━ 4s 4s/step - dice_coeff: 0.5867 - iou_metric: 0.4151 - loss: 0.4133 - val_dice_coeff: 0.5045 - va

 ## Exercise3:Conv3DBlockandEngineeringDiscipline

In [6]:
import mlflow
import numpy as np
from tensorflow import keras

def simple_conv3d_block(input_shape=(32, 32, 32, 1)):
    # Simple block for demonstration: D x H x W x C
    inputs = keras.Input(input_shape)

    # First Conv3D block
    x = keras.layers.Conv3D(
        16, (3, 3, 3), activation='relu', padding='same'
    )(inputs)
    x = keras.layers.MaxPool3D((2, 2, 2))(x)

    # Second Conv3D block (TODO completed)
    x = keras.layers.Conv3D(
        32, (3, 3, 3), activation='relu', padding='same'
    )(x)
    x = keras.layers.MaxPool3D((2, 2, 2))(x)

    x = keras.layers.Flatten()(x)
    outputs = keras.layers.Dense(1, activation='sigmoid')(x)  # Dummy output

    return keras.Model(inputs, outputs)

if __name__ == "__main__":
    mlflow.set_experiment("3D_Volumetric_Analysis")

    with mlflow.start_run(run_name="Conv3D_Baseline"):
        model_3d = simple_conv3d_block()

        # Log Architecture (Engineering Practice)
        model_config = model_3d.to_json()
        mlflow.log_dict(
            {"model_config": model_config},
            "artifacts/model_architecture.json"
        )

        # Log Hyperparameters
        mlflow.log_param("optimizer", "adam")
        mlflow.log_param("filters_start", 16)

        # Simulate training and log metrics (TODO completed)
        final_val_loss = 0.45
        mlflow.log_metric("final_val_loss", final_val_loss)

        print("MLflow tracking complete for 3D block experiment.")


2025/12/26 08:37:04 INFO mlflow.store.db.utils: Creating initial MLflow database tables...
2025/12/26 08:37:04 INFO mlflow.store.db.utils: Updating database tables
2025/12/26 08:37:04 INFO alembic.runtime.migration: Context impl SQLiteImpl.
2025/12/26 08:37:04 INFO alembic.runtime.migration: Will assume non-transactional DDL.
2025/12/26 08:37:04 INFO alembic.runtime.migration: Running upgrade  -> 451aebb31d03, add metric step
2025/12/26 08:37:04 INFO alembic.runtime.migration: Running upgrade 451aebb31d03 -> 90e64c465722, migrate user column to tags
2025/12/26 08:37:04 INFO alembic.runtime.migration: Running upgrade 90e64c465722 -> 181f10493468, allow nulls for metric values
2025/12/26 08:37:04 INFO alembic.runtime.migration: Running upgrade 181f10493468 -> df50e92ffc5e, Add Experiment Tags Table
2025/12/26 08:37:04 INFO alembic.runtime.migration: Running upgrade df50e92ffc5e -> 7ac759974ad8, Update run tags with larger limit
2025/12/26 08:37:04 INFO alembic.runtime.migration: Running 

MLflow tracking complete for 3D block experiment.
